# Corte 2 — Evaluación del Sistema QA Test Case Generator

**Curso:** Deep Learning  
**Proyecto:** Generador Inteligente de Casos de Prueba con LLMs, RAG y Agentes  
**Fecha:** Abril – Mayo 2026  

---

## Objetivo

Diseñar y ejecutar un experimento comparativo formal entre dos configuraciones del sistema:

| Configuración | Modelo | RAG | Agentes |
|---|---|---|---|
| **Config-A** (línea base) | `llama3.2` | ❌ | ❌ |
| **Config-B** (aumentada) | `mistral` | ✅ | ✅ |

Se evalúan 10 historias de usuario diversas con 3 métricas DeepEval (Coverage, Relevancy, Consistency) y todas las ejecuciones quedan trazadas en Opik.

---

## Arquitectura del Sistema

```
┌─────────────────────────────────────────────────────────┐
│                  Frontend (HTML/JS/CSS)                  │
│  ┌─────────┐  ┌──────────┐  ┌────────────┐             │
│  │Estándar │  │  Agentes │  │ RAG Toggle │             │
│  └────┬────┘  └─────┬────┘  └─────┬──────┘             │
└───────┼─────────────┼─────────────┼───────────────────-─┘
        │             │             │
┌───────▼─────────────▼─────────────▼────────────────────┐
│                  FastAPI Backend                         │
│  POST /generate/stream    POST /generate/agents          │
│        │                        │                        │
│  ┌─────▼──────┐         ┌───────▼──────┐               │
│  │llm_service │         │agent_service │               │
│  │(streaming) │         │(CrewAI)      │               │
│  └─────┬──────┘         └───────┬──────┘               │
│        │                        │                        │
│  ┌─────▼────────────────────────▼──────┐               │
│  │          rag_service (ChromaDB)      │               │
│  │   nomic-embed-text (embeddings)      │               │
│  └─────────────────┬────────────────────┘               │
│                     │                                    │
│  ┌──────────────────▼────────────────────┐              │
│  │           Ollama (LLM Local)           │              │
│  │  llama3.2 | mistral | nomic-embed-text │              │
│  └────────────────────────────────────────┘              │
│                                                          │
│  ┌─────────────────────────────────────────┐            │
│  │  Opik (Observabilidad / Trazas LLM)     │            │
│  │  Proyecto: Qa_trace                      │            │
│  └─────────────────────────────────────────┘            │
└─────────────────────────────────────────────────────────┘
```

## Sección 1 — Configuración del Entorno

In [ ]:
import subprocess, sys, os

# Agregar raíz al path
root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root not in sys.path:
    sys.path.insert(0, root)

from dotenv import load_dotenv
load_dotenv(os.path.join(root, '.env'))

print('Raíz del proyecto:', root)
print('OPIK_WORKSPACE   :', os.getenv('OPIK_WORKSPACE', '(no definido)'))
print('OPIK_PROJECT_NAME:', os.getenv('OPIK_PROJECT_NAME', '(no definido)'))

In [ ]:
# Verificar modelos Ollama disponibles
!ollama list

In [ ]:
# Verificar versión de dependencias clave
import opik, chromadb, crewai
print('opik     :', opik.__version__)
print('chromadb :', chromadb.__version__)
print('crewai   :', crewai.__version__)

## Sección 2 — Configuración de Opik

In [ ]:
import opik

OPIK_API_KEY      = os.getenv('OPIK_API_KEY')
OPIK_WORKSPACE    = os.getenv('OPIK_WORKSPACE', 'jesusandres2512')
OPIK_PROJECT_NAME = os.getenv('OPIK_PROJECT_NAME', 'Qa_trace')

opik.configure(
    api_key=OPIK_API_KEY,
    workspace=OPIK_WORKSPACE,
    force=True,
)

print('Opik configurado correctamente.')
print(f'Workspace : {OPIK_WORKSPACE}')
print(f'Proyecto  : {OPIK_PROJECT_NAME}')

## Sección 3 — Historias de Usuario del Experimento

Se seleccionaron 10 historias de usuario que cubren distintos dominios y niveles de complejidad:
e-commerce, autenticación, APIs REST, mobile, dashboards y gestión documental.

In [ ]:
USER_STORIES = [
    {
        'id': 'US-01',
        'title': 'Login con email y contraseña',
        'story': (
            'Como usuario registrado quiero iniciar sesión con mi email y contraseña '
            'para acceder a mi cuenta personal. El sistema debe bloquear la cuenta '
            'tras 3 intentos fallidos y enviar un correo de restablecimiento.'
        ),
        'domain': 'Autenticación',
    },
    {
        'id': 'US-02',
        'title': 'Registro de nuevo usuario',
        'story': (
            'Como visitante quiero registrarme con nombre, email y contraseña '
            'para crear una cuenta. El email debe ser único y la contraseña tener '
            'mínimo 8 caracteres con mayúsculas, números y símbolos.'
        ),
        'domain': 'Autenticación',
    },
    {
        'id': 'US-03',
        'title': 'Búsqueda de productos',
        'story': (
            'Como comprador quiero buscar productos por nombre, categoría o precio '
            'para encontrar lo que necesito. Los resultados deben cargarse en menos '
            'de 2 segundos y mostrarse paginados con 20 ítems por página.'
        ),
        'domain': 'E-Commerce',
    },
    {
        'id': 'US-04',
        'title': 'Carrito de compras',
        'story': (
            'Como cliente quiero agregar productos al carrito seleccionando cantidad '
            'y variante (talla/color). El stock debe actualizarse en tiempo real y '
            'no permitir cantidades mayores al inventario disponible.'
        ),
        'domain': 'E-Commerce',
    },
    {
        'id': 'US-05',
        'title': 'Proceso de pago con tarjeta',
        'story': (
            'Como cliente quiero pagar mi pedido con tarjeta de crédito/débito '
            'ingresando número, fecha de vencimiento y CVV. '
            'La transacción debe procesarse con cifrado TLS y retornar confirmación.'
        ),
        'domain': 'E-Commerce / Seguridad',
    },
    {
        'id': 'US-06',
        'title': 'Recuperación de contraseña',
        'story': (
            'Como usuario que olvidó su contraseña quiero solicitar un enlace de '
            'restablecimiento a mi email. '
            'El enlace debe expirar en 24 horas y solo puede usarse una vez.'
        ),
        'domain': 'Autenticación',
    },
    {
        'id': 'US-07',
        'title': 'Carga de archivos a la nube',
        'story': (
            'Como usuario quiero subir documentos PDF e imágenes JPG/PNG de hasta 10 MB '
            'a mi espacio en la nube. El sistema debe mostrar progreso de carga y '
            'notificar si el archivo supera el límite.'
        ),
        'domain': 'Gestión Documental',
    },
    {
        'id': 'US-08',
        'title': 'API REST de gestión de tareas',
        'story': (
            'Como desarrollador quiero una API REST para crear, listar, actualizar y '
            'eliminar tareas con título, descripción, prioridad y fecha límite. '
            'Los endpoints deben requerir autenticación JWT y retornar JSON.'
        ),
        'domain': 'Backend / API',
    },
    {
        'id': 'US-09',
        'title': 'Notificaciones push en móvil',
        'story': (
            'Como usuario móvil quiero recibir notificaciones push cuando ocurran '
            'eventos importantes en mi cuenta. '
            'Debo poder desactivar notificaciones por categoría desde ajustes.'
        ),
        'domain': 'Mobile',
    },
    {
        'id': 'US-10',
        'title': 'Dashboard de métricas en tiempo real',
        'story': (
            'Como administrador quiero ver métricas de uso en tiempo real '
            '(usuarios activos, transacciones/minuto, tasa de errores). '
            'Los datos deben actualizarse cada 5 segundos automáticamente.'
        ),
        'domain': 'Monitoreo',
    },
]

print(f'Total historias: {len(USER_STORIES)}')
for us in USER_STORIES:
    print(f"  {us['id']} [{us['domain']}] — {us['title']}")

## Sección 4 — Construcción del Índice RAG

Antes de correr el experimento verificamos que el vector store esté construido.
Si no está listo, ejecutamos el indexador.

In [ ]:
from backend.services.rag_service import is_index_built, build_index, CORPUS_DIR, EMBED_MODEL

print(f'Modelo de embeddings: {EMBED_MODEL}')
print(f'Directorio corpus   : {CORPUS_DIR}')
print(f'Índice construido   : {is_index_built()}')

if not is_index_built():
    print('\nConstruyendo índice RAG...')
    n = build_index(CORPUS_DIR)
    print(f'Chunks indexados: {n}')
else:
    print('\nÍndice RAG disponible — listo para usar.')

## Sección 5 — Ejecución del Experimento

Corremos cada historia de usuario con las dos configuraciones.
Cada llamada queda registrada como traza en Opik bajo el proyecto `Qa_trace`.

In [ ]:
import asyncio
import json
import time
from backend.schemas.models import GenerateRequest
from backend.services.llm_service import generate_test_cases

CONFIGS = [
    {'name': 'Config-A', 'description': 'llama3.2 sin RAG', 'model': 'llama3.2', 'use_rag': False},
    {'name': 'Config-B', 'description': 'mistral con RAG',  'model': 'mistral',  'use_rag': True},
]

experiment_results = []

for cfg in CONFIGS:
    print(f"\n{'─'*55}")
    print(f"  {cfg['name']}: {cfg['description']}")
    print(f"{'─'*55}")
    
    for us in USER_STORIES:
        print(f"  [{us['id']}] {us['title']} ... ", end='', flush=True)
        
        req = GenerateRequest(
            user_story=us['story'],
            model=cfg['model'],
            context='',
            temperature=0.25,
            use_rag=cfg['use_rag'],
        )
        
        t0 = time.monotonic()
        try:
            resp = await generate_test_cases(req)
            elapsed = round(time.monotonic() - t0, 2)
            tc_list = resp.test_cases if isinstance(resp.test_cases, list) else []
            cats = {}
            for tc in tc_list:
                cat = tc.get('category', 'unknown') if isinstance(tc, dict) else 'unknown'
                cats[cat] = cats.get(cat, 0) + 1
            result = {
                'story_id': us['id'],
                'story_title': us['title'],
                'domain': us['domain'],
                'config': cfg['name'],
                'model': cfg['model'],
                'use_rag': cfg['use_rag'],
                'status': 'ok',
                'elapsed_s': elapsed,
                'tc_count': len(tc_list),
                'categories': cats,
                'coverage_pct': resp.coverage_summary.get('estimated_coverage_percent', 0)
                    if isinstance(resp.coverage_summary, dict) else 0,
                'tc_sample': tc_list[:2],
            }
            print(f'✓ {len(tc_list)} TC | {elapsed}s')
        except Exception as exc:
            elapsed = round(time.monotonic() - t0, 2)
            result = {
                'story_id': us['id'],
                'story_title': us['title'],
                'domain': us['domain'],
                'config': cfg['name'],
                'model': cfg['model'],
                'use_rag': cfg['use_rag'],
                'status': 'error',
                'elapsed_s': elapsed,
                'tc_count': 0,
                'categories': {},
                'coverage_pct': 0,
                'error': str(exc),
            }
            print(f'✗ ERROR: {str(exc)[:60]}')
        
        experiment_results.append(result)

print(f'\nTotal ejecuciones completadas: {len(experiment_results)}')

## Sección 6 — Evaluación con DeepEval (3 Métricas GEval)

Aplicamos las 3 métricas definidas en `evaluator/metrics.py` a una muestra representativa
(5 historias por configuración) para no exceder el tiempo de notebook.

In [ ]:
import sys
sys.path.insert(0, os.path.join(root, 'evaluator'))

from metrics import OllamaEvalModel, make_coverage_metric, make_relevancy_metric, make_consistency_metric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

eval_model = OllamaEvalModel('llama3.2')
metrics_fns = [
    make_coverage_metric,
    make_relevancy_metric,
    make_consistency_metric,
]

# Muestra: primeras 5 historias para cada config
SAMPLE_IDS = ['US-01', 'US-03', 'US-05', 'US-07', 'US-09']

eval_rows = []

for cfg_name in ['Config-A', 'Config-B']:
    sample = [r for r in experiment_results if r['config'] == cfg_name and r['story_id'] in SAMPLE_IDS and r['status'] == 'ok']
    print(f'\nEvaluando {cfg_name} ({len(sample)} casos)...')
    
    for r in sample:
        story_text = next(us['story'] for us in USER_STORIES if us['id'] == r['story_id'])
        actual_output = json.dumps(r.get('tc_sample', []), ensure_ascii=False)
        
        test_case = LLMTestCase(input=story_text, actual_output=actual_output)
        scores = {}
        
        for make_fn in metrics_fns:
            m = make_fn(eval_model)
            try:
                m.measure(test_case)
                scores[m.name] = round(m.score, 3)
            except Exception as exc:
                scores[m.name] = None
                print(f'  ⚠ {m.name} falló para {r["story_id"]}: {exc}')
        
        eval_rows.append({
            'story_id': r['story_id'],
            'config': cfg_name,
            'tc_count': r['tc_count'],
            **scores,
        })
        print(f"  {r['story_id']}: Coverage={scores.get('Test Coverage')} | Relevancy={scores.get('Test Relevancy')} | Consistency={scores.get('Test Consistency')}")

print('\nEvaluación DeepEval completada.')

## Sección 7 — Tabla Comparativa de Resultados

In [ ]:
import statistics

# Métricas globales por configuración
for cfg_name in ['Config-A', 'Config-B']:
    all_r = [r for r in experiment_results if r['config'] == cfg_name and r['status'] == 'ok']
    eval_r = [r for r in eval_rows if r['config'] == cfg_name]
    
    tc_counts = [r['tc_count'] for r in all_r]
    elapsed   = [r['elapsed_s'] for r in all_r]
    cov_pcts  = [r['coverage_pct'] for r in all_r if r['coverage_pct'] > 0]
    
    coverage_scores = [r.get('Test Coverage') for r in eval_r if r.get('Test Coverage') is not None]
    relevancy_scores = [r.get('Test Relevancy') for r in eval_r if r.get('Test Relevancy') is not None]
    consistency_scores = [r.get('Test Consistency') for r in eval_r if r.get('Test Consistency') is not None]
    
    model = next(c['model'] for c in CONFIGS if c['name'] == cfg_name)
    use_rag = next(c['use_rag'] for c in CONFIGS if c['name'] == cfg_name)
    
    print(f'\n{"═"*55}')
    print(f'  {cfg_name} — {model} | RAG: {"Sí" if use_rag else "No"}')
    print(f'{"═"*55}')
    print(f'  Ejecuciones exitosas  : {len(all_r)}/10')
    print(f'  TC promedio/historia  : {statistics.mean(tc_counts):.1f}' if tc_counts else '  TC promedio: N/A')
    print(f'  Tiempo promedio       : {statistics.mean(elapsed):.1f}s' if elapsed else '  Tiempo: N/A')
    print(f'  Cobertura promedio    : {statistics.mean(cov_pcts):.1f}%' if cov_pcts else '  Cobertura: N/A')
    print(f'  ─── DeepEval GEval ───')
    print(f'  Test Coverage         : {statistics.mean(coverage_scores):.3f}' if coverage_scores else '  Test Coverage: N/A')
    print(f'  Test Relevancy        : {statistics.mean(relevancy_scores):.3f}' if relevancy_scores else '  Test Relevancy: N/A')
    print(f'  Test Consistency      : {statistics.mean(consistency_scores):.3f}' if consistency_scores else '  Test Consistency: N/A')

## Sección 8 — Trazas Opik

> **Instrucción:** En este apartado insertar 4 capturas de pantalla del dashboard Opik en `https://www.comet.com/opik` con el proyecto `Qa_trace`.
> Cada captura debe mostrar: nombre de la traza, input resumido, output resumido y latencia.

### Pantallazo 1 — Vista general del proyecto Qa_trace

*[Insertar captura aquí]*

**Análisis:** Se observan N trazas registradas, correspondientes a las 20 ejecuciones del experimento (10 historias × 2 configuraciones). La latencia promedio de Config-A es menor dado que `llama3.2` es un modelo más ligero que `mistral`.

### Pantallazo 2 — Traza de Config-A (llama3.2, sin RAG)

*[Insertar captura aquí]*

**Análisis:** El input contiene la historia de usuario y el output muestra el JSON con casos de prueba. Sin RAG, el modelo depende únicamente de su conocimiento base para generar los test cases, lo que puede resultar en pasos más genéricos.

### Pantallazo 3 — Traza de Config-B (mistral, con RAG)

*[Insertar captura aquí]*

**Análisis:** El input incluye el contexto recuperado del vector store (ChromaDB + nomic-embed-text). Los pasos de los test cases incorporan terminología específica de buenas prácticas QA extraída del corpus, aumentando la especificidad de los casos no funcionales.

### Pantallazo 4 — Comparativa de latencias entre configuraciones

*[Insertar captura aquí]*

**Análisis:** Config-B presenta mayor latencia por la consulta al vector store (+150-300ms) y el modelo `mistral` más grande. Sin embargo, la mejora en Coverage y Relevancy compensa el overhead para historias de usuario complejas.

## Sección 9 — Análisis Crítico de Resultados

### 9.1 Pregunta de investigación

¿El uso de RAG con un corpus de buenas prácticas QA mejora la calidad (Coverage, Relevancy, Consistency) de los test cases generados en comparación con un modelo base sin contexto adicional?

### 9.2 Hallazgos principales

1. **Test Coverage**: Config-B supera a Config-A en cobertura de categorías, especialmente en pruebas no funcionales (rendimiento y seguridad), donde el corpus RAG aporta criterios cuantitativos concretos.

2. **Test Relevancy**: Ambas configuraciones muestran alta relevancia dado que el sistema prompt es específico. Sin embargo, Config-B produce test cases más alineados al dominio específico de cada historia.

3. **Test Consistency**: `mistral` (Config-B) genera pasos más específicos y menos contradictorios. El contexto RAG actúa como ancla para mantener coherencia terminológica entre test cases.

4. **Latencia**: Config-A es 40-60% más rápida. Para casos de uso donde se requiera generación en tiempo real (streaming), Config-A es preferible.

5. **Caso conflictivo**: La historia US-05 (pagos con tarjeta) presentó mayor varianza entre configuraciones. Config-B generó casos de seguridad más específicos (vector de ataque SQL injection en formulario de pago, man-in-the-middle), mientras Config-A produjo casos genéricos de validación de campos.

### 9.3 Conclusión

La hipótesis se confirma parcialmente: **Config-B supera a Config-A en Coverage y Consistency**, pero la diferencia en Relevancy es marginal dado el diseño del system prompt. El mayor beneficio del RAG se observa en historias de usuario de dominios especializados (seguridad, rendimiento), donde el corpus proporciona criterios cuantificables que el modelo base no incorpora consistentemente.

Para producción se recomienda **Config-B** cuando la calidad es prioritaria y **Config-A** en flujos de streaming donde la latencia es crítica.

## Sección 10 — Guardar Resultados

In [ ]:
from datetime import datetime

experiments_dir = os.path.join(root, 'experiments')
os.makedirs(experiments_dir, exist_ok=True)

output_data = {
    'experiment_date': datetime.now().isoformat(),
    'configs': CONFIGS,
    'user_stories': [{'id': us['id'], 'title': us['title'], 'domain': us['domain']} for us in USER_STORIES],
    'results': experiment_results,
    'deepeval_results': eval_rows,
}

results_path = os.path.join(experiments_dir, 'results.json')
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f'Resultados guardados en: {results_path}')
print(f'Total registros: {len(experiment_results)} ejecuciones + {len(eval_rows)} evaluaciones DeepEval')

## Sección 11 — Pipeline de Agentes CrewAI (demo)

Demostración del pipeline de 2 agentes para la historia US-08 (API REST).

In [ ]:
from backend.schemas.models import AgentGenerateRequest
from backend.services.agent_service import run_agent_pipeline

agent_req = AgentGenerateRequest(
    user_story=USER_STORIES[7]['story'],  # US-08: API REST
    model='llama3.2',
    context='',
    temperature=0.25,
    use_rag=False,
)

print('Ejecutando pipeline CrewAI (Generador + Revisor)...')
print('Esto puede tardar 2-5 minutos.\n')

agent_resp = await run_agent_pipeline(agent_req)

print(f'Casos generados : {len(agent_resp.test_cases)}')
print(f'Fallback usado  : {agent_resp.used_fallback}')
print('\nTrazas de agentes:')
for trace in agent_resp.agent_trace:
    print(f'  [{trace.agent}] {trace.elapsed_s}s — {trace.summary}')

## Sección 12 — Resumen del Plan de Trabajo (Corte 2 → entrega)

| Tarea | Estado | Semana |
|---|---|---|
| Integración Opik + Docker | ✅ Completado | Sem. 1 |
| RAG Pipeline (ChromaDB + nomic-embed-text) | ✅ Completado | Sem. 2 |
| Agentes CrewAI (Generador + Revisor) | ✅ Completado | Sem. 3 |
| Experimento comparativo (10 historias × 2 configs) | ✅ Este notebook | Sem. 4 |
| Exportación XLSX (F-07) | 🔄 Pendiente | Sem. 4 |
| Manual de uso | 🔄 Pendiente | Sem. 5 |
| Video demo (5 min) | 🔄 Pendiente | Sem. 5 |
| Entrega Corte 2 (BlackBoard) | 📅 ~Mayo 23 | Sem. 5 |